# Core-score sensitivity and robustness scaffold

## 1. Purpose and scope

This notebook is a validation-oriented scaffold for consolidating core-score sensitivity and robustness checks that are currently spread across:

- `notebooks/04_directed_results_top30.ipynb`
- `notebooks/04_directed_results_plus3.ipynb`
- `notebooks/overlap_confirmation.ipynb`
- robustness sections in `notebooks/04_directed_results.ipynb`

For this initial scaffold, the notebook intentionally avoids full recomputation, scientific calculation changes, manuscript figure generation, and duplicated analysis blocks. Future work should use this notebook to validate or regenerate namespaced sensitivity artifacts without replacing the canonical manuscript workflow.

## 2. Inputs and artifact policy

Planned inputs should remain read-only unless a future implementation explicitly adds a controlled regeneration step. The notebook may inspect existing project artifacts, including prior sensitivity files and canonical core-score inputs, but it must not alter source data or manuscript outputs.

Artifact policy for this notebook:

- Read from existing notebooks and results only as needed for validation.
- Do not write to `results/figures/paper/`.
- Do not create canonical manuscript outputs.
- Do not create DOCX manuscript tables.
- Keep any future generated files under `results/sensitivity/` with variant-specific subdirectories.

In [ ]:
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
SENSITIVITY_DIR = PROJECT_ROOT / "results" / "sensitivity"

SOURCE_NOTEBOOKS = {
    "canonical": PROJECT_ROOT / "notebooks" / "04_directed_results.ipynb",
    "top30": PROJECT_ROOT / "notebooks" / "04_directed_results_top30.ipynb",
    "strict_plus3": PROJECT_ROOT / "notebooks" / "04_directed_results_plus3.ipynb",
    "overlap_confirmation": PROJECT_ROOT / "notebooks" / "overlap_confirmation.ipynb",
}

EXPECTED_SENSITIVITY_ARTIFACTS = {
    "top30_scores": SENSITIVITY_DIR / "top30" / "core_scores_top30.csv",
    "top100_scores": SENSITIVITY_DIR / "top100" / "core_scores_top100.csv",
    "sensitivity_readme": SENSITIVITY_DIR / "README.md",
}

PROJECT_ROOT, SENSITIVITY_DIR

## 3. Sensitivity variants

Initial variant registry:

| variant | top_n | min_votes | target_up | target_dn | notes |
|---|---:|---:|---:|---:|---|
| `canonical_top50` | 50 | 2 | 42 | 35 | Canonical reference settings used for comparison. |
| `top30` | 30 | 2 | 42 | 35 | Reduced ranked-gene threshold sensitivity check. |
| `top100` | 100 | 2 | 42 | 35 | Expanded ranked-gene threshold sensitivity check. |
| `strict_plus3` | 50 | 3 | 18 | 6 | Stricter voting threshold sensitivity check. |

The registry below is metadata only. It should be used by future validation code to avoid hard-coding variant parameters in multiple places.

In [ ]:
VARIANTS = {
    "canonical_top50": {
        "top_n": 50,
        "min_votes": 2,
        "target_up": 42,
        "target_dn": 35,
    },
    "top30": {
        "top_n": 30,
        "min_votes": 2,
        "target_up": 42,
        "target_dn": 35,
    },
    "top100": {
        "top_n": 100,
        "min_votes": 2,
        "target_up": 42,
        "target_dn": 35,
    },
    "strict_plus3": {
        "top_n": 50,
        "min_votes": 3,
        "target_up": 18,
        "target_dn": 6,
    },
}

VARIANTS

## 4. Planned validation checks

Future implementation should add lightweight validation checks before any regeneration logic. Candidate checks:

1. Confirm required source notebooks and input artifacts exist.
2. Confirm all variant parameters match the registry above.
3. Confirm sensitivity artifacts are written only under `results/sensitivity/`.
4. Compare regenerated sensitivity tables with existing artifacts when present.
5. Validate expected columns, row counts, target up/down counts, and stable sort keys.
6. Confirm no canonical manuscript figure or DOCX outputs are created by this notebook.
7. Summarize pass/fail status in notebook output without changing scientific calculations.

In [ ]:
artifact_status = {
    "source_notebooks": {
        name: path.exists() for name, path in SOURCE_NOTEBOOKS.items()
    },
    "expected_sensitivity_artifacts": {
        name: path.exists() for name, path in EXPECTED_SENSITIVITY_ARTIFACTS.items()
    },
}

artifact_status

## 4A. Historical and namespaced artifact inventory

Some sensitivity artifacts already have namespaced replacements under `results/sensitivity/`, including the `top30` and `top100` core-score CSV files. Other artifacts remain historical-only notebook-root files that are retained as fallback or provenance references.

This notebook currently performs validation only. The inventory below records preferred namespaced paths where available and historical fallback paths where artifacts have not yet been migrated. The checks are read-only and do not recompute core scores, regenerate sensitivity artifacts, create figures, or rewrite historical files.

In [ ]:
ARTIFACT_REGISTRY = {
    "top30_scores": {
        "preferred_path": SENSITIVITY_DIR / "top30" / "core_scores_top30.csv",
        "fallback_path": PROJECT_ROOT / "notebooks" / "core_scores_top30.csv",
        "artifact_type": "csv",
        "status_role": "namespaced_preferred_with_historical_fallback",
    },
    "top50_scores": {
        "preferred_path": None,
        "fallback_path": PROJECT_ROOT / "notebooks" / "core_scores_top50.csv",
        "artifact_type": "csv",
        "status_role": "historical_only_reference",
    },
    "top100_scores": {
        "preferred_path": SENSITIVITY_DIR / "top100" / "core_scores_top100.csv",
        "fallback_path": PROJECT_ROOT / "notebooks" / "core_scores_top100.csv",
        "artifact_type": "csv",
        "status_role": "namespaced_preferred_with_historical_fallback",
    },
    "core_v2_pickle": {
        "preferred_path": None,
        "fallback_path": PROJECT_ROOT / "notebooks" / "core_v2.pkl",
        "artifact_type": "pickle",
        "status_role": "historical_only_provenance",
    },
    "core_v3_pickle": {
        "preferred_path": None,
        "fallback_path": PROJECT_ROOT / "notebooks" / "core_v3.pkl",
        "artifact_type": "pickle",
        "status_role": "historical_only_provenance",
    },
    "core_v2_top30_pickle": {
        "preferred_path": None,
        "fallback_path": PROJECT_ROOT / "notebooks" / "core_v2_top30.pkl",
        "artifact_type": "pickle",
        "status_role": "historical_only_top30_provenance",
    },
    "core_v2_top100_pickle": {
        "preferred_path": None,
        "fallback_path": PROJECT_ROOT / "notebooks" / "core_v2_top100.pkl",
        "artifact_type": "pickle",
        "status_role": "historical_only_top100_provenance",
    },
}

ARTIFACT_REGISTRY

In [ ]:
import pandas as pd

artifact_inventory_rows = []
for artifact_name, metadata in ARTIFACT_REGISTRY.items():
    preferred_path = metadata["preferred_path"]
    fallback_path = metadata["fallback_path"]
    preferred_exists = preferred_path.exists() if preferred_path is not None else False
    fallback_exists = fallback_path.exists() if fallback_path is not None else False
    selected_path = preferred_path if preferred_exists else fallback_path if fallback_exists else preferred_path or fallback_path

    artifact_inventory_rows.append(
        {
            "artifact": artifact_name,
            "preferred_exists": preferred_exists,
            "fallback_exists": fallback_exists,
            "selected_path": str(selected_path.relative_to(PROJECT_ROOT)) if selected_path is not None else None,
            "artifact_type": metadata["artifact_type"],
            "status_role": metadata["status_role"],
        }
    )

artifact_inventory = pd.DataFrame(artifact_inventory_rows)
artifact_inventory

### Lightweight CSV schema validation

For CSV artifacts only, the next check validates whether the selected read-only artifact exists and, when present, reports table dimensions and expected-column coverage. Missing artifacts are reported without raising errors.

In [ ]:
EXPECTED_CSV_COLUMNS = {"sig_id", "core_score"}

csv_validation_rows = []
for artifact_name, metadata in ARTIFACT_REGISTRY.items():
    if metadata["artifact_type"] != "csv":
        continue

    preferred_path = metadata["preferred_path"]
    fallback_path = metadata["fallback_path"]
    selected_path = preferred_path if preferred_path is not None and preferred_path.exists() else fallback_path
    exists = selected_path.exists() if selected_path is not None else False

    row_count = None
    column_count = None
    missing_expected_columns = sorted(EXPECTED_CSV_COLUMNS)
    error = None

    if exists:
        try:
            csv_df = pd.read_csv(selected_path)
            row_count = len(csv_df)
            column_count = len(csv_df.columns)
            missing_expected_columns = sorted(EXPECTED_CSV_COLUMNS.difference(csv_df.columns))
        except Exception as exc:
            error = f"{type(exc).__name__}: {exc}"

    csv_validation_rows.append(
        {
            "artifact": artifact_name,
            "exists": exists,
            "selected_path": str(selected_path.relative_to(PROJECT_ROOT)) if selected_path is not None else None,
            "row_count": row_count,
            "column_count": column_count,
            "missing_expected_columns": missing_expected_columns,
            "error": error,
        }
    )

csv_schema_validation = pd.DataFrame(csv_validation_rows)
csv_schema_validation

## 4B. Precomputed robustness correlation checks

These checks validate robustness using existing precomputed artifacts only. They do not recompute core scores, regenerate sensitivity outputs, create figures, or rewrite artifacts.

The read-only comparisons below use the artifact registry to compare precomputed scores for:

- `top30` vs `top50`
- `top100` vs `top50`
- `top30` vs `top100`


In [ ]:
ROBUSTNESS_SCORE_ARTIFACTS = ["top30_scores", "top50_scores", "top100_scores"]


def _path_label(path):
    if path is None:
        return None
    try:
        return str(path.relative_to(PROJECT_ROOT))
    except ValueError:
        return str(path)


def _resolve_registered_artifact(artifact_name):
    metadata = ARTIFACT_REGISTRY.get(artifact_name)
    if metadata is None:
        return None, f"{artifact_name}: not present in ARTIFACT_REGISTRY"

    candidates = [
        ("preferred", metadata.get("preferred_path")),
        ("fallback", metadata.get("fallback_path")),
    ]
    checked = []
    for label, path in candidates:
        exists = path.exists() if path is not None else False
        checked.append(f"{label}={_path_label(path)} exists={exists}")
        if exists:
            return path, None

    return None, f"{artifact_name}: no readable artifact found ({'; '.join(checked)})"


robustness_score_paths = {}
robustness_loader_rows = []
for artifact_name in ROBUSTNESS_SCORE_ARTIFACTS:
    selected_path, warning = _resolve_registered_artifact(artifact_name)
    robustness_score_paths[artifact_name] = selected_path
    robustness_loader_rows.append(
        {
            "artifact": artifact_name,
            "selected_path": _path_label(selected_path),
            "available": selected_path is not None,
            "warning": warning,
        }
    )

robustness_loader_status = pd.DataFrame(robustness_loader_rows)
robustness_loader_status


In [ ]:
ROBUSTNESS_SCORE_COLUMNS = {
    "top30_scores": "score30",
    "top50_scores": "score50",
    "top100_scores": "score100",
}

robustness_score_tables_raw = {}
robustness_score_tables = {}
robustness_merge_messages = []

for artifact_name, score_column in ROBUSTNESS_SCORE_COLUMNS.items():
    selected_path = robustness_score_paths.get(artifact_name)
    if selected_path is None:
        robustness_merge_messages.append(f"{artifact_name}: skipped because no artifact path was resolved")
        continue

    try:
        raw_table = pd.read_csv(selected_path)
    except Exception as exc:
        robustness_merge_messages.append(f"{artifact_name}: could not read CSV ({type(exc).__name__}: {exc})")
        continue

    robustness_score_tables_raw[artifact_name] = raw_table
    required_columns = {"sig_id", "core_score"}
    missing_columns = sorted(required_columns.difference(raw_table.columns))
    if missing_columns:
        robustness_merge_messages.append(
            f"{artifact_name}: skipped because required columns are missing: {missing_columns}"
        )
        continue

    robustness_score_tables[artifact_name] = raw_table[["sig_id", "core_score"]].rename(
        columns={"core_score": score_column}
    )

required_tables_present = all(
    artifact_name in robustness_score_tables for artifact_name in ROBUSTNESS_SCORE_ARTIFACTS
)

if required_tables_present:
    robustness_scores_merged = (
        robustness_score_tables["top30_scores"]
        .merge(robustness_score_tables["top50_scores"], on="sig_id", how="inner")
        .merge(robustness_score_tables["top100_scores"], on="sig_id", how="inner")
    )
else:
    robustness_scores_merged = pd.DataFrame(columns=["sig_id", "score30", "score50", "score100"])

robustness_merge_validation = pd.DataFrame(
    [
        {
            "merged_row_count": len(robustness_scores_merged),
            "duplicate_sig_id_count": int(robustness_scores_merged["sig_id"].duplicated().sum())
            if "sig_id" in robustness_scores_merged.columns
            else None,
            "missing_score30": int(robustness_scores_merged["score30"].isna().sum())
            if "score30" in robustness_scores_merged.columns
            else None,
            "missing_score50": int(robustness_scores_merged["score50"].isna().sum())
            if "score50" in robustness_scores_merged.columns
            else None,
            "missing_score100": int(robustness_scores_merged["score100"].isna().sum())
            if "score100" in robustness_scores_merged.columns
            else None,
            "messages": robustness_merge_messages,
        }
    ]
)
robustness_merge_validation


In [ ]:
ROBUSTNESS_COMPARISONS = [
    ("top30 vs top50", "score30", "score50"),
    ("top100 vs top50", "score100", "score50"),
    ("top30 vs top100", "score30", "score100"),
]


def _score_correlations(frame, comparisons):
    rows = []
    for comparison_label, left_column, right_column in comparisons:
        if frame.empty or left_column not in frame.columns or right_column not in frame.columns:
            rows.append(
                {
                    "comparison": comparison_label,
                    "pearson_r": pd.NA,
                    "spearman_r": pd.NA,
                    "n": 0,
                }
            )
            continue

        paired_scores = frame[[left_column, right_column]].apply(pd.to_numeric, errors="coerce").dropna()
        rows.append(
            {
                "comparison": comparison_label,
                "pearson_r": paired_scores[left_column].corr(paired_scores[right_column], method="pearson")
                if len(paired_scores) >= 2
                else pd.NA,
                "spearman_r": paired_scores[left_column].corr(paired_scores[right_column], method="spearman")
                if len(paired_scores) >= 2
                else pd.NA,
                "n": len(paired_scores),
            }
        )
    return pd.DataFrame(rows, columns=["comparison", "pearson_r", "spearman_r", "n"])


global_robustness_correlations = _score_correlations(
    robustness_scores_merged,
    ROBUSTNESS_COMPARISONS,
)
global_robustness_correlations


### Per-cell-line robustness correlations

When `cell_id` is available in all three precomputed score tables, the next read-only check merges cell-line labels safely and calculates the same correlation summaries within each cell line. If any prerequisite is missing, the check reports a notebook-readable status and returns an empty summary table.


In [ ]:
per_cell_messages = []
per_cell_ready = required_tables_present

for artifact_name in ROBUSTNESS_SCORE_ARTIFACTS:
    raw_table = robustness_score_tables_raw.get(artifact_name)
    if raw_table is None:
        per_cell_ready = False
        per_cell_messages.append(f"{artifact_name}: raw score table is unavailable")
    elif "cell_id" not in raw_table.columns:
        per_cell_ready = False
        per_cell_messages.append(f"{artifact_name}: missing cell_id column")

if per_cell_ready:
    per_cell_inputs = []
    for artifact_name, score_column in ROBUSTNESS_SCORE_COLUMNS.items():
        cell_column = score_column.replace("score", "cell_id")
        per_cell_inputs.append(
            robustness_score_tables_raw[artifact_name][["sig_id", "cell_id", "core_score"]].rename(
                columns={"cell_id": cell_column, "core_score": score_column}
            )
        )

    per_cell_scores_merged = (
        per_cell_inputs[0]
        .merge(per_cell_inputs[1], on="sig_id", how="inner")
        .merge(per_cell_inputs[2], on="sig_id", how="inner")
    )
    cell_id_consistent = (
        (per_cell_scores_merged["cell_id30"] == per_cell_scores_merged["cell_id50"])
        & (per_cell_scores_merged["cell_id30"] == per_cell_scores_merged["cell_id100"])
    )
    mismatched_cell_id_count = int((~cell_id_consistent).sum())
    if mismatched_cell_id_count:
        per_cell_messages.append(
            f"Skipped {mismatched_cell_id_count} merged rows with inconsistent cell_id values across artifacts"
        )
    per_cell_scores_merged = per_cell_scores_merged.loc[cell_id_consistent].rename(
        columns={"cell_id30": "cell_id"}
    )

    per_cell_rows = []
    for cell_id, cell_frame in per_cell_scores_merged.groupby("cell_id", dropna=False):
        cell_summary = _score_correlations(cell_frame, ROBUSTNESS_COMPARISONS)
        cell_summary.insert(0, "cell_id", cell_id)
        per_cell_rows.append(cell_summary)

    per_cell_robustness_correlations = (
        pd.concat(per_cell_rows, ignore_index=True)
        if per_cell_rows
        else pd.DataFrame(columns=["cell_id", "comparison", "pearson_r", "spearman_r", "n"])
    )
else:
    per_cell_scores_merged = pd.DataFrame()
    mismatched_cell_id_count = None
    per_cell_robustness_correlations = pd.DataFrame(
        columns=["cell_id", "comparison", "pearson_r", "spearman_r", "n"]
    )

per_cell_robustness_status = pd.DataFrame(
    [
        {
            "per_cell_check_ran": per_cell_ready,
            "merged_row_count": len(per_cell_scores_merged),
            "mismatched_cell_id_count": mismatched_cell_id_count,
            "messages": per_cell_messages,
        }
    ]
)

per_cell_robustness_status, per_cell_robustness_correlations


## 4C. Read-only gene-set overlap checks

These checks validate existing pickle artifacts only. They do not regenerate gene sets, recompute signatures from expression matrices, create figures, or rewrite artifacts.

This section preserves the useful read-only overlap checks from `overlap_confirmation.ipynb` while routing artifact discovery through `ARTIFACT_REGISTRY` instead of hard-coded pickle paths.

For the overlap summary below, `retained_fraction_of_left` is `overlap_size / left_size`, and `retained_fraction_of_right` is `overlap_size / right_size`. This makes the denominator convention explicit for both the canonical/strict comparison and the top100/top30 comparison.


In [ ]:
import pickle

PICKLE_GENE_SET_ARTIFACTS = [
    "core_v2_pickle",
    "core_v3_pickle",
    "core_v2_top30_pickle",
    "core_v2_top100_pickle",
]


def _load_registered_gene_set_pickle(artifact_name):
    selected_path, warning = _resolve_registered_artifact(artifact_name)
    result = {
        "artifact": artifact_name,
        "selected_path": _path_label(selected_path),
        "available": selected_path is not None,
        "valid": False,
        "up": None,
        "down": None,
        "status_message": warning,
    }

    if selected_path is None:
        if result["status_message"] is None:
            result["status_message"] = f"{artifact_name}: no artifact path was resolved"
        return result

    try:
        with selected_path.open("rb") as handle:
            loaded_object = pickle.load(handle)
    except Exception as exc:
        result["status_message"] = f"{artifact_name}: could not load pickle ({type(exc).__name__}: {exc})"
        return result

    if not isinstance(loaded_object, tuple):
        result["status_message"] = f"{artifact_name}: expected tuple, observed {type(loaded_object).__name__}"
        return result

    if len(loaded_object) != 2:
        result["status_message"] = f"{artifact_name}: expected tuple of length 2, observed length {len(loaded_object)}"
        return result

    up_set, down_set = loaded_object
    if not isinstance(up_set, set) or not isinstance(down_set, set):
        result["status_message"] = (
            f"{artifact_name}: expected both tuple elements to be sets, "
            f"observed {type(up_set).__name__} and {type(down_set).__name__}"
        )
        return result

    result.update(
        {
            "valid": True,
            "up": up_set,
            "down": down_set,
            "status_message": "valid tuple of UP and DOWN sets loaded read-only",
        }
    )
    return result


gene_set_pickle_loads = {
    artifact_name: _load_registered_gene_set_pickle(artifact_name)
    for artifact_name in PICKLE_GENE_SET_ARTIFACTS
}

gene_set_pickle_validation = pd.DataFrame(
    [
        {
            "artifact": artifact_name,
            "selected_path": load_result["selected_path"],
            "available": load_result["available"],
            "valid": load_result["valid"],
            "up_size": len(load_result["up"]) if load_result["up"] is not None else pd.NA,
            "down_size": len(load_result["down"]) if load_result["down"] is not None else pd.NA,
            "status_message": load_result["status_message"],
        }
        for artifact_name, load_result in gene_set_pickle_loads.items()
    ]
)

gene_set_pickle_validation


In [ ]:
GENE_SET_OVERLAP_COMPARISONS = [
    ("canonical/core_v2 vs strict/core_v3", "core_v2_pickle", "core_v3_pickle"),
    ("top100 vs top30", "core_v2_top100_pickle", "core_v2_top30_pickle"),
]


def _gene_set_overlap_row(comparison, left_artifact, right_artifact, direction):
    left_load = gene_set_pickle_loads[left_artifact]
    right_load = gene_set_pickle_loads[right_artifact]
    base_row = {
        "comparison": comparison,
        "direction": direction,
        "left_artifact": left_artifact,
        "right_artifact": right_artifact,
        "left_size": pd.NA,
        "right_size": pd.NA,
        "overlap_size": pd.NA,
        "retained_fraction_of_left": pd.NA,
        "retained_fraction_of_right": pd.NA,
        "status_message": None,
    }

    status_messages = []
    if not left_load["valid"]:
        status_messages.append(left_load["status_message"])
    if not right_load["valid"]:
        status_messages.append(right_load["status_message"])
    if status_messages:
        base_row["status_message"] = "; ".join(message for message in status_messages if message)
        return base_row

    left_set = left_load[direction]
    right_set = right_load[direction]
    overlap_size = len(left_set & right_set)
    left_size = len(left_set)
    right_size = len(right_set)

    base_row.update(
        {
            "left_size": left_size,
            "right_size": right_size,
            "overlap_size": overlap_size,
            "retained_fraction_of_left": overlap_size / left_size if left_size else pd.NA,
            "retained_fraction_of_right": overlap_size / right_size if right_size else pd.NA,
            "status_message": (
                "ok; retained_fraction_of_left uses overlap_size/left_size; "
                "retained_fraction_of_right uses overlap_size/right_size"
            ),
        }
    )
    return base_row


gene_set_overlap_validation = pd.DataFrame(
    [
        _gene_set_overlap_row(comparison, left_artifact, right_artifact, direction)
        for comparison, left_artifact, right_artifact in GENE_SET_OVERLAP_COMPARISONS
        for direction in ("up", "down")
    ],
    columns=[
        "comparison",
        "direction",
        "left_artifact",
        "right_artifact",
        "left_size",
        "right_size",
        "overlap_size",
        "retained_fraction_of_left",
        "retained_fraction_of_right",
        "status_message",
    ],
)

gene_set_overlap_validation


### Overlap validation scope

These checks are historical/provenance validations. They currently use historical notebook-root pickle artifacts through the registry, and they should be replaced by namespaced pickle artifacts only after those artifacts are intentionally created and validated.

These read-only checks do not imply that historical pickle files can be deleted yet.


## Validation-only guarantees

- No scientific recomputation is performed.
- No sensitivity artifacts are regenerated.
- No manuscript outputs are modified.
- This notebook currently validates existing artifacts only.

## 5. Output policy

All future outputs from this notebook must be namespaced under `results/sensitivity/`.

This notebook must not write to canonical manuscript figure directories. Paper figures remain generated by `notebooks/04_directed_results.ipynb` only. The role of this notebook is to validate or regenerate sensitivity artifacts for robustness review, not to replace the canonical manuscript notebook.

## 6. TODO / future implementation steps

- Add read-only loaders for canonical and sensitivity core-score artifacts.
- Add variant-specific validation helpers that consume the `VARIANTS` registry.
- Add checks for expected schema, target counts, vote thresholds, and ranked-gene cutoffs.
- Add optional regeneration behind an explicit flag that defaults to disabled.
- Ensure regeneration writes only to `results/sensitivity/<variant>/`.
- Keep manuscript figure generation and DOCX table generation out of this notebook.

## 7. Validation summary

This table is a review aid. It does not write files. It does not replace manuscript outputs. PASS/WARN/FAIL is operational validation status, not a scientific conclusion.


In [ ]:
import pandas as pd


def _validation_object(name):
    return globals().get(name)


def _validation_detail(messages):
    cleaned = [str(message) for message in messages if message not in (None, "")]
    return "; ".join(cleaned) if cleaned else "No issues reported."


def _missing_object_row(category, *object_names):
    return {
        "category": category,
        "status": "FAIL",
        "summary": "Expected validation output is not available in memory.",
        "detail": f"Run the earlier notebook section(s) that create: {', '.join(object_names)}.",
    }


validation_summary_rows = []

# 1. Source notebooks
_source_status = _validation_object("artifact_status")
if not isinstance(_source_status, dict) or "source_notebooks" not in _source_status:
    validation_summary_rows.append(_missing_object_row("source_notebooks", "artifact_status"))
else:
    source_notebooks_status = _source_status["source_notebooks"]
    missing_sources = [name for name, exists in source_notebooks_status.items() if not exists]
    validation_summary_rows.append(
        {
            "category": "source_notebooks",
            "status": "PASS" if not missing_sources else "FAIL",
            "summary": f"Checked {len(source_notebooks_status)} source notebook path(s).",
            "detail": _validation_detail([f"Missing: {', '.join(missing_sources)}" if missing_sources else None]),
        }
    )

# 2. Sensitivity artifacts
_artifact_inventory = _validation_object("artifact_inventory")
if not isinstance(_artifact_inventory, pd.DataFrame):
    validation_summary_rows.append(_missing_object_row("sensitivity_artifacts", "artifact_inventory"))
else:
    missing_artifacts = _artifact_inventory.loc[
        ~(_artifact_inventory["preferred_exists"] | _artifact_inventory["fallback_exists"]), "artifact"
    ].tolist()
    fallback_only = _artifact_inventory.loc[
        (~_artifact_inventory["preferred_exists"]) & _artifact_inventory["fallback_exists"], "artifact"
    ].tolist()
    validation_summary_rows.append(
        {
            "category": "sensitivity_artifacts",
            "status": "FAIL" if missing_artifacts else "WARN" if fallback_only else "PASS",
            "summary": f"Inventoried {len(_artifact_inventory)} registered artifact(s).",
            "detail": _validation_detail(
                [
                    f"Missing required artifact(s): {', '.join(missing_artifacts)}" if missing_artifacts else None,
                    f"Historical fallback/provenance artifact(s) in use: {', '.join(fallback_only)}" if fallback_only else None,
                ]
            ),
        }
    )

# 3. CSV schema
_csv_schema_validation = _validation_object("csv_schema_validation")
if not isinstance(_csv_schema_validation, pd.DataFrame):
    validation_summary_rows.append(_missing_object_row("csv_schema", "csv_schema_validation"))
else:
    missing_csv = _csv_schema_validation.loc[~_csv_schema_validation["exists"], "artifact"].tolist()
    malformed_csv = _csv_schema_validation.loc[
        _csv_schema_validation["error"].notna()
        | _csv_schema_validation["missing_expected_columns"].apply(lambda value: len(value) > 0),
        "artifact",
    ].tolist()
    validation_summary_rows.append(
        {
            "category": "csv_schema",
            "status": "FAIL" if missing_csv or malformed_csv else "PASS",
            "summary": f"Validated schema for {len(_csv_schema_validation)} CSV artifact(s).",
            "detail": _validation_detail(
                [
                    f"Missing CSV artifact(s): {', '.join(missing_csv)}" if missing_csv else None,
                    f"Malformed or schema-incomplete CSV artifact(s): {', '.join(malformed_csv)}" if malformed_csv else None,
                ]
            ),
        }
    )

# 4. Robustness merge
_robustness_loader_status = _validation_object("robustness_loader_status")
_robustness_merge_validation = _validation_object("robustness_merge_validation")
if not isinstance(_robustness_loader_status, pd.DataFrame) or not isinstance(_robustness_merge_validation, pd.DataFrame):
    validation_summary_rows.append(
        _missing_object_row("robustness_merge", "robustness_loader_status", "robustness_merge_validation")
    )
else:
    unavailable_scores = _robustness_loader_status.loc[~_robustness_loader_status["available"], "artifact"].tolist()
    merge_row = _robustness_merge_validation.iloc[0].to_dict() if len(_robustness_merge_validation) else {}
    merge_messages = merge_row.get("messages") or []
    merge_failures = []
    if merge_row.get("merged_row_count", 0) == 0:
        merge_failures.append("merged_row_count is 0")
    for column in ("duplicate_sig_id_count", "missing_score30", "missing_score50", "missing_score100"):
        if merge_row.get(column, 0):
            merge_failures.append(f"{column}={merge_row.get(column)}")
    validation_summary_rows.append(
        {
            "category": "robustness_merge",
            "status": "FAIL" if unavailable_scores or merge_failures or merge_messages else "PASS",
            "summary": f"Merged row count: {merge_row.get('merged_row_count', 'not available')}.",
            "detail": _validation_detail(
                [
                    f"Unavailable score artifact(s): {', '.join(unavailable_scores)}" if unavailable_scores else None,
                    *merge_failures,
                    *merge_messages,
                ]
            ),
        }
    )

# 5. Global correlations
_global_correlations = _validation_object("global_robustness_correlations")
if not isinstance(_global_correlations, pd.DataFrame):
    validation_summary_rows.append(_missing_object_row("global_correlations", "global_robustness_correlations"))
else:
    incomplete_global = _global_correlations.loc[
        (_global_correlations["n"] < 2)
        | _global_correlations[["pearson_r", "spearman_r"]].isna().any(axis=1),
        "comparison",
    ].tolist()
    validation_summary_rows.append(
        {
            "category": "global_correlations",
            "status": "FAIL" if incomplete_global else "PASS",
            "summary": f"Reviewed {len(_global_correlations)} global correlation comparison(s).",
            "detail": _validation_detail(
                [f"Incomplete comparison(s): {', '.join(incomplete_global)}" if incomplete_global else None]
            ),
        }
    )

# 6. Per-cell correlations
_per_cell_status = _validation_object("per_cell_robustness_status")
_per_cell_correlations = _validation_object("per_cell_robustness_correlations")
if not isinstance(_per_cell_status, pd.DataFrame) or not isinstance(_per_cell_correlations, pd.DataFrame):
    validation_summary_rows.append(
        _missing_object_row("per_cell_correlations", "per_cell_robustness_status", "per_cell_robustness_correlations")
    )
else:
    per_cell_row = _per_cell_status.iloc[0].to_dict() if len(_per_cell_status) else {}
    per_cell_ran = bool(per_cell_row.get("per_cell_check_ran", False))
    per_cell_messages = per_cell_row.get("messages") or []
    incomplete_per_cell = []
    if per_cell_ran and len(_per_cell_correlations):
        incomplete_per_cell = _per_cell_correlations.loc[
            (_per_cell_correlations["n"] < 2)
            | _per_cell_correlations[["pearson_r", "spearman_r"]].isna().any(axis=1),
            "comparison",
        ].unique().tolist()
    validation_summary_rows.append(
        {
            "category": "per_cell_correlations",
            "status": "WARN" if not per_cell_ran else "FAIL" if incomplete_per_cell else "PASS",
            "summary": (
                f"Per-cell check {'ran' if per_cell_ran else 'was skipped'}; "
                f"{len(_per_cell_correlations)} comparison row(s) available."
            ),
            "detail": _validation_detail(
                [
                    *per_cell_messages,
                    f"Incomplete per-cell comparison(s): {', '.join(incomplete_per_cell)}" if incomplete_per_cell else None,
                ]
            ),
        }
    )

# 7. Pickle validation
_pickle_validation = _validation_object("gene_set_pickle_validation")
if not isinstance(_pickle_validation, pd.DataFrame):
    validation_summary_rows.append(_missing_object_row("pickle_validation", "gene_set_pickle_validation"))
else:
    invalid_pickles = _pickle_validation.loc[~(_pickle_validation["available"] & _pickle_validation["valid"]), "artifact"].tolist()
    historical_pickles = _pickle_validation.loc[
        _pickle_validation["selected_path"].fillna("").str.startswith("notebooks/"), "artifact"
    ].tolist()
    validation_summary_rows.append(
        {
            "category": "pickle_validation",
            "status": "FAIL" if invalid_pickles else "WARN" if historical_pickles else "PASS",
            "summary": f"Validated {len(_pickle_validation)} gene-set pickle artifact(s).",
            "detail": _validation_detail(
                [
                    f"Invalid or unavailable pickle artifact(s): {', '.join(invalid_pickles)}" if invalid_pickles else None,
                    f"Historical notebook-root pickle artifact(s): {', '.join(historical_pickles)}" if historical_pickles else None,
                ]
            ),
        }
    )

# 8. Gene-set overlap
_overlap_validation = _validation_object("gene_set_overlap_validation")
if not isinstance(_overlap_validation, pd.DataFrame):
    validation_summary_rows.append(_missing_object_row("gene_set_overlap", "gene_set_overlap_validation"))
else:
    failed_overlap = _overlap_validation.loc[
        _overlap_validation["overlap_size"].isna()
        | ~_overlap_validation["status_message"].fillna("").str.startswith("ok"),
        ["comparison", "direction"],
    ]
    failed_overlap_labels = [f"{row.comparison} ({row.direction})" for row in failed_overlap.itertuples()]
    validation_summary_rows.append(
        {
            "category": "gene_set_overlap",
            "status": "FAIL" if failed_overlap_labels else "WARN",
            "summary": f"Reviewed {len(_overlap_validation)} overlap validation row(s).",
            "detail": _validation_detail(
                [
                    f"Incomplete overlap row(s): {', '.join(failed_overlap_labels)}" if failed_overlap_labels else None,
                    "Historical/provenance overlap check; replace with namespaced pickle artifacts only after intentional migration.",
                ]
            ),
        }
    )

# 9. Output safety policy
validation_summary_rows.append(
    {
        "category": "output_safety_policy",
        "status": "PASS",
        "summary": "Summary construction is in-memory and validation-only.",
        "detail": "This cell does not write files, regenerate artifacts, create figures, or change manuscript outputs.",
    }
)

validation_summary = pd.DataFrame(
    validation_summary_rows,
    columns=["category", "status", "summary", "detail"],
)
validation_summary
